Data preprocessing so we have a simple .csv file that can be used for model training

In [19]:
import pandas as pd
import os
import numpy as np
from pathlib import Path

raw_dir = Path("../data/raw/events")

# functions for variable calculations
# doing some angle calculations
def calc_angle(x, y):
    goal_width = 7.32

    numerator = goal_width * x
    denominator = (x**2 + y**2 - (goal_width**2 / 4))

    return np.arctan2(numerator, denominator)


for file_path in raw_dir.glob("*.json"):
    event = pd.read_json(
        file_path
    )

    # to help with labelling final processed csv
    match_id = file_path.stem

    # filtering for shots
    shots = event[event["type"].apply(lambda x: x["name"] == "Shot")].copy()

    # Skip matches with no shots (important safety check)
    if shots.empty:
        continue

    # creating "y" for our label (goal: y/n)
    shots["goal"] = shots["shot"].apply(
        lambda x: x["outcome"]["name"] == "Goal"
    )

    # extracting coordinates of ball
    shots["x"] = shots["location"].apply(lambda x: x[0])
    shots["y"] = shots["location"].apply(lambda x: x[1])

    # calculating distance
    shots["distance"] = np.sqrt(
        (120 - shots["x"])**2 +
        (40 - shots["y"])**2
    )

    shots["angle"] = shots.apply(
        lambda r: calc_angle(r["x"], r["y"]),
        axis=1
    )

    # categorical features (shot type and body part used)
    shots["body_part"] = shots["shot"].apply(
        lambda x: x["body_part"]["name"]
    )

    shots["shot_type"] = shots["shot"].apply(
        lambda x: x["type"]["name"]
    )

    # cleaning up for final dataset
    data = shots[[
        "distance",
        "angle",
        "body_part",
        "shot_type",
        "goal"
    ]].dropna()

    # encoding
    data = pd.get_dummies(data, columns=["body_part", "shot_type"])

    # final csv
    output_path = f"../data/processed/shots_{match_id}.csv"
    data.to_csv(output_path, index=False)

For angle calculations:


![Angle calculations](angle_calc.png)

Columns for outputted .csv:

distance

angle

goal

body_part_Head

body_part_Left Foot

body_part_Right Foot

shot_type_Free Kick

shot_type_Open Play

shot_type_Penalty

In [21]:
# checking to see if # of raw event files = # of processed files
from pathlib import Path

dir_path = Path("../data/raw/events")

num_files_raw = len(list(dir_path.glob("*.json")))
print("Raw: ", num_files_raw)


dir_path = Path("../data/processed")

num_files_proc = len(list(dir_path.glob("*.csv")))
print("Processed: ", num_files_proc)


Raw:  4235
Processed:  4235
